In [1]:
# baseline_01_naive_previous_day.py
# Naive baseline: predict same hour previous day (t-24) per home_id
# Outputs: metrics CSV (overall + per-home) and optional predictions CSV

import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# Paths
# =========================
BASE_DIR = Path("C:/IDEAL_Programming")
DATA_PATH = BASE_DIR / "processed" / "final" / "IDEAL_final_hourly_dataset.csv"

OUT_DIR = BASE_DIR / "processed" / "baselines" / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_METRICS_ALL = OUT_DIR / "baseline01_naive_prevday_overall.csv"
OUT_METRICS_HOME = OUT_DIR / "baseline01_naive_prevday_per_home.csv"
OUT_PREDS = OUT_DIR / "baseline01_naive_prevday_predictions.csv"

# =========================
# Settings
# =========================
TARGET = "consumption_Wh"
TIME_COL = "timestamp"
HOME_COL = "home_id"

# If True, saves row-level predictions (can be large)
SAVE_PREDICTIONS = True

# =========================
# Helpers: metrics
# =========================
def mae(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mape(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true), eps) # avoid division by ~0
    return 100.0 * np.mean(np.abs((y_true - y_pred) / denom))

# =========================
# Load
# =========================
df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL])

# Basic sanity
needed = {HOME_COL, TIME_COL, TARGET}
missing = needed - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Sort
df = df.sort_values([HOME_COL, TIME_COL]).reset_index(drop=True)

# =========================
# Naive prediction: y_hat(t) = y(t-24) per home
# =========================
# Works if your data is hourly. If there are gaps, shift(24) still means "24 rows back",
# so we additionally check that timestamp difference == 24 hours.
df["y_true"] = df[TARGET]
df["y_hat_raw"] = df.groupby(HOME_COL)[TARGET].shift(24)

df["t_prev"] = df.groupby(HOME_COL)[TIME_COL].shift(24)
df["dt_hours"] = (df[TIME_COL] - df["t_prev"]).dt.total_seconds() / 3600.0

# Keep only valid prev-day matches (exactly 24 hours)
df["y_hat"] = np.where(df["dt_hours"] == 24.0, df["y_hat_raw"], np.nan)

# Keep evaluable rows
eval_df = df.dropna(subset=["y_true", "y_hat"]).copy()

print("Rows total:", len(df))
print("Rows evaluable (have prev-day):", len(eval_df))
print("Unique homes:", df[HOME_COL].nunique())

# =========================
# Overall metrics
# =========================
overall = {
    "baseline": "naive_prev_day_same_hour",
    "rows_evaluable": int(len(eval_df)),
    "MAE": float(mae(eval_df["y_true"], eval_df["y_hat"])),
    "RMSE": float(rmse(eval_df["y_true"], eval_df["y_hat"])),
    "MAPE_%": float(mape(eval_df["y_true"], eval_df["y_hat"])),
}
overall_df = pd.DataFrame([overall])
overall_df.to_csv(OUT_METRICS_ALL, index=False)

# =========================
# Per-home metrics
# =========================
per_home_rows = []
for home_id, g in eval_df.groupby(HOME_COL):
    per_home_rows.append({
        "home_id": home_id,
        "rows_evaluable": int(len(g)),
        "MAE": float(mae(g["y_true"], g["y_hat"])),
        "RMSE": float(rmse(g["y_true"], g["y_hat"])),
        "MAPE_%": float(mape(g["y_true"], g["y_hat"])),
    })

per_home_df = pd.DataFrame(per_home_rows).sort_values("MAE", ascending=True)
per_home_df.to_csv(OUT_METRICS_HOME, index=False)

# =========================
# Optional: save predictions
# =========================
if SAVE_PREDICTIONS:
    preds = eval_df[[HOME_COL, TIME_COL, "y_true", "y_hat"]].copy()
    preds.to_csv(OUT_PREDS, index=False)

print("\nSaved:")
print("Overall metrics:", OUT_METRICS_ALL)
print("Per-home metrics:", OUT_METRICS_HOME)
if SAVE_PREDICTIONS:
    print("Predictions:", OUT_PREDS)

print("\nOverall metrics:")
print(overall_df)

C:\Users\tsift\AppData\Local\Temp\ipykernel_8436\154445090.py:54: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL])


Rows total: 1529350
Rows evaluable (have prev-day): 1170358
Unique homes: 254

Saved:
Overall metrics: C:\IDEAL_Programming\processed\baselines\results\baseline01_naive_prevday_overall.csv
Per-home metrics: C:\IDEAL_Programming\processed\baselines\results\baseline01_naive_prevday_per_home.csv
Predictions: C:\IDEAL_Programming\processed\baselines\results\baseline01_naive_prevday_predictions.csv

Overall metrics:
                   baseline  rows_evaluable         MAE        RMSE  \
0  naive_prev_day_same_hour         1170358  223.703961  461.911093   

         MAPE_%  
0  1.393576e+06  
